In [ ]:
!nvidia-smi
%pip install torchmetrics[image] -q

In [ ]:
from __future__ import annotations
from typing import Callable, Iterable, Optional

import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor
from torch.optim import Optimizer

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance

In [ ]:
SEED = 42 # For robust reporting, run multiple seeds: 42, 6, 11, 96, 22
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
class HNAdam(Optimizer):
    def __init__(
        self,
        params: Iterable[Tensor],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.99),
        eps: float = 1e-8,
        lambda_t0: Optional[float] = None,
    ) -> None:
        if params is None:
            raise ValueError("params cannot be None.")
        if lr <= 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if eps < 0.0:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if len(betas) != 2:
            raise ValueError("betas must be a tuple of two floats")

        beta1, beta2 = betas
        if not 0.0 <= beta1 < 1.0:
            raise ValueError(f"Invalid beta1 value: {beta1}")
        if not 0.0 <= beta2 < 1.0:
            raise ValueError(f"Invalid beta2 value: {beta2}")

        if lambda_t0 is None:
            lambda_t0 = random.uniform(2.0, 4.0)
        if not 2.0 <= lambda_t0 <= 4.0:
            raise ValueError(f"lambda_t0 must be in [2, 4], got {lambda_t0}")

        defaults = {
            "lr": lr,
            "betas": (beta1, beta2),
            "eps": eps,
            "lambda_t0": lambda_t0,
            "amsgrad": False,
        }
        super().__init__(params, defaults)

        if len(self.param_groups) == 0:
            raise ValueError("optimizer got an empty parameter list")

    @torch.no_grad()
    def step(self, closure: Optional[Callable[[], Tensor]] = None) -> Optional[Tensor]:
        loss: Optional[Tensor] = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr: float = group["lr"]
            beta1, beta2 = group["betas"]
            eps: float = group["eps"]
            lambda_t0: float = group["lambda_t0"]

            for param in group["params"]:
                if param.grad is None:
                    continue

                grad = param.grad
                if grad.is_sparse:
                    raise RuntimeError("HNAdam does not support sparse gradients")

                state = self.state[param]

                if len(state) == 0:
                    state["m"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["v"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["vhat"] = torch.zeros_like(param, memory_format=torch.preserve_format)

                m_prev: Tensor = state["m"]
                v_prev: Tensor = state["v"]
                vhat_prev: Tensor = state["vhat"]

                g_t = grad

                m_t = beta1 * m_prev + (1.0 - beta1) * g_t

                g_abs = g_t.abs()
                m_prev_norm = torch.linalg.vector_norm(m_prev)
                g_abs_norm = torch.linalg.vector_norm(g_abs)
                m_max = torch.maximum(m_prev_norm, g_abs_norm)

                zero = torch.zeros((), dtype=param.dtype, device=param.device)
                ratio = torch.where(m_max > 0.0, m_prev_norm / m_max, zero)
                lambda_t = torch.as_tensor(lambda_t0, dtype=param.dtype, device=param.device) - ratio

                v_t = beta2 * v_prev + (1.0 - beta2) * g_abs.pow(lambda_t)

                if bool((lambda_t < 2.0).item()):
                    group["amsgrad"] = True

                    vhat_t = torch.maximum(vhat_prev, v_t.abs())
                    state["vhat"] = vhat_t

                    denom = vhat_t.pow(1.0 / lambda_t) + eps
                else:
                    group["amsgrad"] = False

                    denom = v_t.pow(1.0 / lambda_t) + eps

                param.addcdiv_(m_t, denom, value=-lr)

                state["m"] = m_t
                state["v"] = v_t

        return loss

In [ ]:
class Adam(Optimizer):
    """
    Implements Adam algorithm.
    
    This implementation strictly adheres to the pseudocode provided in the 
    official PyTorch documentation and the original paper (Kingma & Ba, 2014).
    """

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.0, amsgrad=False, maximize=False):
        
        # Handle edge cases and invalid configurations
        if params is None:
            raise ValueError("Invalid params: None")
        if betas is None or not isinstance(betas, (tuple, list)) or len(betas) != 2:
            raise ValueError("Invalid betas: expected a tuple of two floats")
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= eps:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 0: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta parameter at index 1: {betas[1]}")
        if not 0.0 <= weight_decay:
            raise ValueError(f"Invalid weight_decay value: {weight_decay}")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay,
                        amsgrad=amsgrad, maximize=maximize)
        super(Adam, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        """Performs a single optimization step."""
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            # Extract hyperparameters for the group
            lr = group['lr']
            beta1, beta2 = group['betas']
            eps = group['eps']
            weight_decay = group['weight_decay']
            amsgrad = group['amsgrad']
            maximize = group['maximize']

            for p in group['params']:
                # Skip parameters with no gradients
                if p.grad is None:
                    continue
                
                if p.grad.is_sparse:
                    raise RuntimeError("Adam does not support sparse gradients.")

                # g_t = grad(theta_{t-1})
                grad = p.grad
                
                state = self.state[p]

                # State initialization
                if len(state) == 0:
                    state['step'] = 0
                    # m_0 = 0 (Exponential moving average of gradient values)
                    state['exp_avg'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    # v_0 = 0 (Exponential moving average of squared gradient values)
                    state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    if amsgrad:
                        # v_max_0 = 0 (Maintains max of all exp. moving avg. of sq. grad. values)
                        state['max_exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

                # t = t + 1
                state['step'] += 1
                t = state['step']

                exp_avg = state['exp_avg']
                exp_avg_sq = state['exp_avg_sq']

                g_t = grad

                # if maximize: g_t = -g_t
                if maximize:
                    g_t = -g_t

                # if weight_decay != 0: g_t = g_t + lambda * theta_{t-1}
                if weight_decay != 0:
                    g_t = g_t.add(p, alpha=weight_decay)

                # m_t = beta_1 * m_{t-1} + (1 - beta_1) * g_t
                exp_avg.mul_(beta1).add_(g_t, alpha=1.0 - beta1)

                # v_t = beta_2 * v_{t-1} + (1 - beta_2) * g_t^2
                exp_avg_sq.mul_(beta2).addcmul_(g_t, g_t, value=1.0 - beta2)

                # if amsgrad: v_hat_t = max(v_hat_{t-1}, v_t)
                if amsgrad:
                    max_exp_avg_sq = state['max_exp_avg_sq']
                    # v_t^max = max(v_t^max, v_t)
                    torch.maximum(max_exp_avg_sq, exp_avg_sq, out=max_exp_avg_sq)
                    # v_hat_t = v_t^max
                    v_hat_t = max_exp_avg_sq
                # else: v_hat_t = v_t
                else:
                    # v_hat_t = v_t
                    v_hat_t = exp_avg_sq

                # bias_correction1 = 1 - beta_1^t
                bias_correction1 = 1.0 - beta1 ** t
                
                # bias_correction2 = 1 - beta_2^t
                bias_correction2 = 1.0 - beta2 ** t

                # m_hat_t = m_t / (1 - beta_1^t)
                m_hat_t = exp_avg / bias_correction1
                # v_hat_t = v_hat_t / (1 - beta_2^t)
                v_hat_t = v_hat_t / bias_correction2

                # theta_t = theta_{t-1} - lr * m_hat_t / (sqrt(v_hat_t) + eps)
                denom = v_hat_t.sqrt().add_(eps)
                p.addcdiv_(m_hat_t, denom, value=-lr)

        return loss

In [ ]:
class SGD(Optimizer):
    """
    Implements Stochastic Gradient Descent (optionally with momentum, Nesterov, and weight decay).
    
    This implementation strictly translates the official PyTorch 2.12.0 pseudocode logic
    into an efficient, executable Optimizer format.
    """
    def __init__(
        self, 
        params: Iterable[torch.Tensor], 
        lr: float, 
        momentum: float = 0.0, 
        dampening: float = 0.0,
        weight_decay: float = 0.0, 
        nesterov: bool = False, 
        maximize: bool = False
    ):
        # Handle edge cases: Validate hyperparameter inputs
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if momentum < 0.0:
            raise ValueError(f"Invalid momentum value: {momentum}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay value: {weight_decay}")
        if dampening < 0.0:
            raise ValueError(f"Invalid dampening value: {dampening}")
        if nesterov and (momentum <= 0.0 or dampening != 0.0):
            raise ValueError("Nesterov momentum requires a positive momentum and zero dampening")

        defaults = dict(
            lr=lr, momentum=momentum, dampening=dampening,
            weight_decay=weight_decay, nesterov=nesterov, maximize=maximize
        )
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure: Optional[Callable] = None) -> Optional[float]:
        """Performs a single optimization step."""
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        # for t=1 to ... do (Implied loop driven by successive calls to optimizer.step())
        for group in self.param_groups:
            weight_decay = group['weight_decay']
            momentum = group['momentum']
            dampening = group['dampening']
            nesterov = group['nesterov']
            maximize = group['maximize']
            lr = group['lr']

            for p in group['params']:
                # Handle edge case: Skip parameter if there is no gradient available
                if p.grad is None:
                    continue
                    
                # if maximize:
                #     g_t = -gradient
                # else:
                #     g_t = gradient
                g_t = -p.grad if maximize else p.grad

                # if g_t is sparse: only the plain update is supported
                if g_t.is_sparse:
                    if weight_decay != 0.0 or momentum != 0.0 or nesterov:
                        raise RuntimeError(
                            "Sparse gradients do not support weight_decay, momentum, or nesterov"
                        )
                    # theta_t = theta_{t-1} - lr * g_t
                    p.add_(g_t, alpha=-lr)
                    continue

                # if weight_decay != 0:
                #     g_t = g_t + weight_decay * theta_{t-1}
                if weight_decay != 0.0:
                    g_t = g_t.add(p, alpha=weight_decay)

                # if momentum != 0:
                if momentum != 0.0:
                    param_state = self.state[p]
                    
                    # if t > 1:
                    #     b_t = momentum * b_{t-1} + (1 - dampening) * g_t
                    # else:
                    #     b_t = g_t
                    if 'momentum_buffer' not in param_state:
                        # Initialization at t=1: b_t = g_t
                        b_t = param_state['momentum_buffer'] = torch.clone(g_t).detach()
                    else:
                        # Subsequent steps t > 1: momentum calculation
                        b_t = param_state['momentum_buffer']
                        # Modifies buffer in-place for maximum efficiency
                        b_t.mul_(momentum).add_(g_t, alpha=1.0 - dampening)

                    # if nesterov:
                    #     g_t = g_t + momentum * b_t
                    # else:
                    #     g_t = b_t
                    if nesterov:
                        g_t = g_t.add(b_t, alpha=momentum)
                    else:
                        g_t = b_t

                # theta_t = theta_{t-1} - lr * g_t
                p.add_(g_t, alpha=-lr)

        return loss

In [ ]:
# Hyperparameters
IMAGE_CHANNELS = 3
IMAGE_HEIGHT = 96
IMAGE_WIDTH = 96
input_dim = IMAGE_CHANNELS * IMAGE_HEIGHT * IMAGE_WIDTH  # 3 x 96 x 96 RGB images flattened for balanced training and better reconstruction quality
hidden_dim = 500
latent_dim = 20
BATCH_SIZE = 128
EPOCHS = 50
lr = 1e-3
betas = (0.9, 0.99)
eps = 1e-8

# Data loader
transform = transforms.Compose([
    transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)),
    transforms.Lambda(lambda img: img.convert('RGB')),
    transforms.ToTensor(),
])

full_dataset = datasets.Caltech101(
    root='./data',
    target_type='category',
    transform=transform,
    download=True,
 )

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
split_generator = torch.Generator().manual_seed(SEED)
train_dataset, test_dataset = torch.utils.data.random_split(
    full_dataset,
    [train_size, test_size],
    generator=split_generator,
 )

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

class_names = full_dataset.categories

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples:  {len(test_dataset)}')

# Display one sample image from each of the first 10 classes
fig, axes = plt.subplots(1, 10, figsize=(18, 3))
class_samples = {}

for imgs, labels in train_loader:
    for i in range(imgs.size(0)):
        label = labels[i].item()
        if label not in class_samples and label < 10:
            class_samples[label] = imgs[i]
        if len(class_samples) == 10:
            break
    if len(class_samples) == 10:
        break

display_labels = sorted(class_samples.keys())
for idx, label in enumerate(display_labels):
    axes[idx].imshow(np.transpose(class_samples[label].cpu().numpy(), (1, 2, 0)))
    axes[idx].set_title(class_names[label], fontsize=9)
    axes[idx].axis('off')

for idx in range(len(display_labels), 10):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Define the Encoder, Decoder, and VAE
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = torch.tanh(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()
        self.fc1 = nn.Linear(latent_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        h = torch.tanh(self.fc1(z))
        x_hat = torch.sigmoid(self.fc2(h))
        return x_hat


class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, hidden_dim, input_dim)

    def reparameterize(self, mu, logvar):
        # Reparameterization trick: z = mu + epsilon * std
        std = torch.exp(0.5 * logvar)
        epsilon = torch.randn_like(std)
        z = mu + epsilon * std
        return z

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar


def build_model():
    return VAE(input_dim, hidden_dim, latent_dim).to(device)


build_model()

In [ ]:
# Define the loss function
def vae_loss(x, x_hat, mu, logvar):
    # Reconstruction term
    mse = F.mse_loss(x_hat, x, reduction='sum')

    # KL divergence term
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    return mse + kld


optimizer_builders = {
    'HNAdam': lambda params: HNAdam(params, lr=lr, betas=betas, eps=eps),
    'Adam': lambda params: Adam(params, lr=lr, betas=betas, eps=eps, amsgrad=False),
    'AMSGrad': lambda params: Adam(params, lr=lr, betas=betas, eps=eps, amsgrad=True),
    'SGD': lambda params: SGD(params, lr=lr),
}

print('Comparing optimizers:', ', '.join(optimizer_builders.keys()))

In [ ]:
histories = {}
models = {}
training_times = {}


def train_model(model, optimizer, train_loader, epochs):
    history = []
    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        running_total = 0.0

        for x, _ in train_loader:
            x = x.to(device)
            x_flat = x.view(x.size(0), -1)

            optimizer.zero_grad()
            x_hat, mu, logvar = model(x_flat)
            total_loss = vae_loss(x_flat, x_hat, mu, logvar)
            if isinstance(optimizer, SGD): # Only apply gradient clipping for SGD to prevent divergence
                (total_loss / x.size(0)).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            else:
                total_loss.backward()
            optimizer.step()

            running_total += total_loss.item()

        avg_total = running_total / len(train_loader.dataset)
        history.append(avg_total)
        print(f'Epoch {epoch:02d}/{epochs} | total={avg_total:.4f}')

    training_time_sec = time.perf_counter() - start_time
    return history, training_time_sec


for optimizer_name, optimizer_builder in optimizer_builders.items():
    print(f'\n===== Training with {optimizer_name} =====')
    model = build_model()
    optimizer = optimizer_builder(model.parameters())
    history, training_time_sec = train_model(model, optimizer, train_loader, EPOCHS)

    models[optimizer_name] = model
    histories[optimizer_name] = history
    training_times[optimizer_name] = training_time_sec

    print(f'{optimizer_name} completed in {training_time_sec:.2f} seconds')

In [ ]:
# Figure 1: loss minimization curves
epochs = np.arange(1, EPOCHS + 1)

plt.figure(figsize=(10, 6))
for optimizer_name, loss_history in histories.items():
    plt.plot(epochs, loss_history, label=optimizer_name, linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('Loss (average per sample)')
plt.title('Figure 1. VAE Loss Function Minimization Curves by Optimizer')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
@torch.no_grad()
def show_reconstructions(model, data_loader, title, n_images=10):
    model.eval()
    x, _ = next(iter(data_loader))
    x = x.to(device)
    x_flat = x.view(x.size(0), -1)

    x_prob, _, _ = model(x_flat)
    x_recon = x_prob.view(-1, IMAGE_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH)

    n = min(n_images, x.size(0))
    _, axes = plt.subplots(2, n, figsize=(1.8 * n, 3.6))

    for i in range(n):
        axes[0, i].imshow(np.transpose(x[i].cpu().numpy(), (1, 2, 0)))
        axes[0, i].axis('off')
        axes[1, i].imshow(np.transpose(x_recon[i].cpu().numpy(), (1, 2, 0)))
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=10)
    axes[1, 0].set_ylabel('Reconstructed', fontsize=10)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def show_generated_samples(model, latent_dim, title, n_images=64):
    model.eval()
    z = torch.randn(n_images, latent_dim, device=device)
    samples = model.decoder(z).view(-1, IMAGE_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH).clamp(0, 1)

    grid = make_grid(samples.cpu(), nrow=8, pad_value=1.0)
    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis('off')
    plt.title(title)
    plt.show()


for optimizer_name, model in models.items():
    show_reconstructions(model, test_loader, title=f'{optimizer_name}: Original vs Reconstructed Caltech101 images', n_images=10)
    show_generated_samples(model, latent_dim=latent_dim, title=f'{optimizer_name}: Generated Caltech101 samples from latent prior', n_images=64)

In [ ]:
@torch.no_grad()
def preprocess_for_fid(images):
    # FID expects 3-channel uint8 images resized to 299x299.
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)
    if images.size(2) != 299 or images.size(3) != 299:
        images = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)
    images = (images.clamp(0, 1) * 255).to(torch.uint8)
    return images


@torch.no_grad()
def evaluate_ssim_and_fid(model, test_loader, latent_dim=20, num_fid_samples=None):
    model.eval()

    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)

    total_test = len(test_loader.dataset)
    if num_fid_samples is None:
        num_fid_samples = total_test

    # Real and reconstructed images for SSIM.
    for x, _ in test_loader:
        x = x.to(device)
        x_flat = x.view(x.size(0), -1)
        x_prob, _, _ = model(x_flat)
        x_recon = x_prob.view(-1, IMAGE_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH)

        ssim_metric.update(x_recon, x)

    ssim_value = float(ssim_metric.compute().item())

    # FID: real test images vs generated images.
    real_seen = 0
    for x, _ in test_loader:
        if real_seen >= num_fid_samples:
            break

        x = x.to(device)
        needed = min(x.size(0), num_fid_samples - real_seen)
        real_batch = x[:needed]
        fid_metric.update(preprocess_for_fid(real_batch), real=True)
        real_seen += needed

    fake_seen = 0
    while fake_seen < num_fid_samples:
        current_bs = min(BATCH_SIZE, num_fid_samples - fake_seen)
        z = torch.randn(current_bs, latent_dim, device=device)
        x_gen = model.decoder(z).view(-1, IMAGE_CHANNELS, IMAGE_HEIGHT, IMAGE_WIDTH).clamp(0, 1)
        fid_metric.update(preprocess_for_fid(x_gen), real=False)
        fake_seen += current_bs

    fid_value = float(fid_metric.compute().item())
    return ssim_value, fid_value


comparison_rows = []
num_fid_samples = len(test_loader.dataset)

for optimizer_name, model in models.items():
    ssim_value, fid_value = evaluate_ssim_and_fid(
        model,
        test_loader,
        latent_dim=latent_dim,
        num_fid_samples=num_fid_samples,
    )

    comparison_rows.append({
        'Model': f'Vanilla VAE x Caltech101 x {optimizer_name}',
        'Optimizer': optimizer_name,
        'Min Training Loss': float(np.min(histories[optimizer_name])),
        'Training Time (s)': float(training_times[optimizer_name]),
        'SSIM': ssim_value,
        'FID': fid_value,
    })

    print(f'{optimizer_name} -> SSIM: {ssim_value:.4f} | FID: {fid_value:.4f}')

comparison_df = pd.DataFrame(comparison_rows)

In [ ]:
# Table 1: summary statistics across optimizers
table_1 = comparison_df.copy()

table_1 = table_1.round({
    'Min Training Loss': 4,
    'Training Time (s)': 2,
    'SSIM': 4,
    'FID': 4
})

optimizer_order = ['HNAdam', 'Adam', 'AMSGrad', 'SGD']
table_1['Optimizer'] = pd.Categorical(table_1['Optimizer'], categories=optimizer_order, ordered=True)
table_1 = table_1.sort_values(by='Optimizer').reset_index(drop=True)

print('Table 1. Training and Evaluation Summary by Optimizer')
display(table_1)